# Train the court-keypoint model

This notebook trains a ResNet-50 regressor to predict 14 tennis-court keypoints. The dataset archive is linked by the original tutorial author; its redistribution licence has not been verified, so the archive and extracted files remain local and are not stored in Git.

## 1. Install the download helper

Run this notebook from the repository root after installing the project's pinned requirements.

In [ ]:
%pip install -q gdown

## 2. Download and validate the dataset

The download is skipped when valid extracted data already exists. Local training data is written under `training/local_data/`, which is ignored by Git.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import gdown

download_directory = Path("training/local_data")
archive_path = download_directory / "tennis_court_det_dataset.zip"
extraction_directory = download_directory / "tennis_court_keypoints"
data_directory = extraction_directory / "data"

required_paths = [
    data_directory / "images",
    data_directory / "data_train.json",
    data_directory / "data_val.json",
]

if not all(path.exists() for path in required_paths):
    download_directory.mkdir(parents=True, exist_ok=True)
    if not archive_path.exists():
        downloaded_path = gdown.download(
            id="1lhAaeQCmk2y440PmagA0KmIVBIysVMwu",
            output=str(archive_path),
            quiet=False,
        )
        if downloaded_path is None:
            raise RuntimeError("Court-keypoint dataset download failed")

    extraction_directory.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive_path) as archive:
        archive.extractall(extraction_directory)

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    formatted_paths = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(f"Dataset is incomplete:\n{formatted_paths}")

print(f"Dataset ready: {data_directory}")

## 3. Build PyTorch datasets

Each image is resized to 224×224. Its 14 `(x, y)` keypoints are scaled to the same coordinate system, producing 28 target values.

In [ ]:
import json

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


class KeypointsDataset(Dataset):
    def __init__(self, image_directory, annotation_path):
        self.image_directory = Path(image_directory)
        self.annotation_path = Path(annotation_path)
        with self.annotation_path.open(encoding="utf-8") as annotations_file:
            self.annotations = json.load(annotations_file)
        if not self.annotations:
            raise ValueError(f"No annotations found in {self.annotation_path}")

        self.transform = transforms.Compose(
            [
                transforms.ToPILImage(),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ]
        )

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        item = self.annotations[index]
        image_path = self.image_directory / f"{item['id']}.png"
        image = cv2.imread(str(image_path))
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")

        height, width = image.shape[:2]
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image)

        keypoints = np.asarray(item["kps"], dtype=np.float32).reshape(-1)
        if keypoints.size != 28:
            raise ValueError(
                f"Expected 28 keypoint values for {image_path}, got {keypoints.size}"
            )
        keypoints[0::2] *= 224.0 / width
        keypoints[1::2] *= 224.0 / height

        return image, torch.from_numpy(keypoints)

In [ ]:
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

train_dataset = KeypointsDataset(
    data_directory / "images",
    data_directory / "data_train.json",
)
validation_dataset = KeypointsDataset(
    data_directory / "images",
    data_directory / "data_val.json",
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
)

print(
    f"Training samples: {len(train_dataset)}; "
    f"validation samples: {len(validation_dataset)}"
)

## 4. Create the model

The final ResNet-50 layer is replaced with 28 outputs: two coordinates for each of the 14 court keypoints.

In [ ]:
def create_keypoint_model(weights=models.ResNet50_Weights.DEFAULT):
    model = models.resnet50(weights=weights)
    model.fc = torch.nn.Linear(model.fc.in_features, 14 * 2)
    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_keypoint_model().to(device)
print(f"Training device: {device}")

## 5. Train and validate

Training reports mean squared error for both the training and validation splits after each epoch.

In [ ]:
def mean_loader_loss(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, keypoints in data_loader:
            images = images.to(device)
            keypoints = keypoints.to(device)
            loss = criterion(model(images), keypoints)
            total_loss += loss.item() * images.size(0)
    return total_loss / len(data_loader.dataset)


def train_model(model, train_loader, validation_loader, device, epochs=20):
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_training_loss = 0.0
        for images, keypoints in train_loader:
            images = images.to(device)
            keypoints = keypoints.to(device)

            optimizer.zero_grad()
            loss = criterion(model(images), keypoints)
            loss.backward()
            optimizer.step()
            total_training_loss += loss.item() * images.size(0)

        training_loss = total_training_loss / len(train_loader.dataset)
        validation_loss = mean_loader_loss(
            model,
            validation_loader,
            criterion,
            device,
        )
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"train_loss={training_loss:.4f}, "
            f"validation_loss={validation_loss:.4f}"
        )

    return model

In [ ]:
model = train_model(
    model,
    train_loader,
    validation_loader,
    device,
    epochs=20,
)

## 6. Save the checkpoint

The pipeline expects the state dictionary at `models/keypoints_model.pth`. Model weights remain local and are ignored by Git.

In [ ]:
checkpoint_path = Path("models/keypoints_model.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print(f"Saved checkpoint: {checkpoint_path}")